# Phân loại và Dự đoán Bệnh Tiểu đường (Diabetes)
## Sử dụng Pima Indians Diabetes Dataset

## 1. Định nghĩa vấn đề (Define Problem)
+ **Mô tả**:
    + Bộ dữ liệu bao gồm 768 mẫu từ các bệnh nhân của người Ấn Độ Pima.
    + Tám đặc điểm y tế được đo từ mỗi bệnh nhân.
    + Mục tiêu: Dự đoán xem bệnh nhân có bị bệnh tiểu đường hay không (phân loại nhị phân).
+ **Dữ liệu vào (Input Features)**:
    + Pregnancies: Số lần mang thai
    + Glucose: Nồng độ glucose trong máu (mg/dL)
    + BloodPressure: Huyết áp tâm trương (mmHg)
    + SkinThickness: Độ dày da tại cơ ba đầu (mm)
    + Insulin: Nồng độ insulin 2 giờ sau (µU/mL)
    + BMI: Chỉ số khối cơ thể (kg/m²)
    + DiabetesPedigreeFunction: Hàm lịch sử bệnh tiểu đường gia đình
    + Age: Tuổi (năm)
+ **Kết quả (Output/Target)**:
    + Outcome: 0 (Không bị bệnh tiểu đường) hoặc 1 (Bị bệnh tiểu đường)

## 2. Nhập thư viện và tải dữ liệu (Import Libraries & Load Data)

In [ ]:
# Nhập các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')
import joblib
import os

# Thiết lập style cho biểu đồ
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

: 

In [ ]:
# Tải dữ liệu từ file CSV
# Đặt tên cột vì file CSV không có header
column_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv('data/pima-indians-diabetes.csv', header=None, names=column_names)

print(f"Kích thước dữ liệu: {df.shape}")
print(f"\n10 dòng đầu tiên:")
print(df.head(10))

## 3. Khám phá và Phân tích dữ liệu (Exploratory Data Analysis - EDA)

### 3.1 Hướng dẫn (Guidance)
+ **Mục tiêu**: Hiểu rõ dữ liệu trước khi xây dựng mô hình
+ **Các bước thực hiện**:
    1. Kiểm tra kích thước và kiểu dữ liệu
    2. Xem thống kê mô tả (mean, std, min, max)
    3. Phân tích phân bố lớp (class distribution)
    4. Tính tương quan giữa các đặc trưng
    5. Vẽ biểu đồ để trực quan hóa
+ **Nhận xét**:
    - Glucose có tương quan cao nhất với Outcome (0.47)
    - Dữ liệu không cân bằng: 65.1% không bệnh, 34.9% có bệnh
    - Một số giá trị 0 bất thường (Glucose=0, BloodPressure=0 không hợp lý y tế)
    - Cần xử lý giá trị bất thường trước huấn luyện

In [ ]:
# Xem thông tin tổng quan về dữ liệu
print("Thông tin dữ liệu:")
print(df.info())
print(f"\n" + "="*80)
print(f"\nThống kê mô tả:")
print(df.describe())
print(f"\n" + "="*80)
print(f"\nGiá trị bị thiếu:")
print(df.isnull().sum())

In [ ]:
# Kiểm tra phân bố lớp (Class Distribution)
print(f"\nPhân bố lớp (Outcome):")
print(df['Outcome'].value_counts())
print(f"\nTỷ lệ phần trăm:")
print(df['Outcome'].value_counts(normalize=True) * 100)

# Vẽ biểu đồ phân bố
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Biểu đồ cột
df['Outcome'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('Phân bố lớp Diabetes', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Kết quả (0: Không bệnh, 1: Có bệnh)')
axes[0].set_ylabel('Số lượng mẫu')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Biểu đồ tròn
labels = ['Không bệnh (0)', 'Có bệnh (1)']
colors = ['skyblue', 'salmon']
df['Outcome'].value_counts().sort_index().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                                 labels=labels, colors=colors, startangle=90)
axes[1].set_title('Tỷ lệ Diabetes', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Phân tích tương quan giữa các biến
correlation_matrix = df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', 
            square=True, linewidths=0.5)
plt.title('Ma trận tương quan giữa các biến', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Hiển thị tương quan với Outcome
print("\nTương quan với Outcome:")
outcome_corr = correlation_matrix['Outcome'].sort_values(ascending=False)
print(outcome_corr)

In [ ]:
# Vẽ biểu đồ phân bố các đặc trưng
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes = axes.ravel()

features = df.columns[:-1]  # Loại bỏ cột Outcome

for idx, feature in enumerate(features):
    axes[idx].hist(df[feature], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Phân bố {feature}', fontweight='bold')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Tần số')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# So sánh các đặc trưng giữa nhóm bệnh và không bệnh
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, feature in enumerate(features):
    df[df['Outcome'] == 0][feature].hist(bins=20, alpha=0.6, label='Không bệnh', ax=axes[idx], color='skyblue')
    df[df['Outcome'] == 1][feature].hist(bins=20, alpha=0.6, label='Có bệnh', ax=axes[idx], color='salmon')
    axes[idx].set_title(f'{feature} theo kết quả', fontweight='bold')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Tần số')
    axes[idx].legend(loc='upper right')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Tiền xử lý dữ liệu (Data Preprocessing)

### 4.1 Hướng dẫn (Guidance)
+ **Bước 1: Xử lý giá trị 0 bất thường**
    - Thay giá trị 0 bằng trung bình (mean) của các giá trị khác 0
    - Áp dụng cho: Glucose, BloodPressure, SkinThickness, Insulin, BMI
    - Công thức: `mean_val = df[col][df[col] != 0].mean()`
+ **Bước 2: Tách Features và Target**
    - X = tất cả cột trừ Outcome
    - y = cột Outcome (0 hoặc 1)
+ **Bước 3: Chia dữ liệu (Train-Test Split)**
    - 80% training, 20% testing
    - Dùng `stratify=y` để bảo đảm tỷ lệ lớp
+ **Bước 4: Chuẩn hóa dữ liệu (Standardization)**
    - Dùng StandardScaler: X_scaled = (X - mean) / std
    - Fit trên training set, transform cả hai tập
+ **Nhận xét**:
    - Chuẩn hóa rất quan trọng cho SVM và Logistic Regression
    - Random Forest không cần nhưng vẫn làm cho thống nhất
    - Fit scaler trên training set, không fit lại trên test set

In [ ]:
# Kiểm tra và xử lý các giá trị 0 không hợp lý
# Các biến như Glucose, BloodPressure, BMI không nên có giá trị 0
print("Số lượng giá trị 0 trong mỗi cột:")
zero_count = (df == 0).sum()
print(zero_count)
print(f"\nTổng số giá trị 0: {zero_count.sum()}")

# Hiển thị các hàng có giá trị 0
print(f"\nPercentage of zeros in each column:")
for col in zero_count.index:
    if col != 'Outcome':
        pct = (zero_count[col] / len(df)) * 100
        print(f"  {col}: {pct:.2f}%")

In [ ]:
# Thay thế giá trị 0 bằng trung bình (ngoại trừ Pregnancies và Outcome)
# Pregnancies có thể là 0, nhưng các biến khác thì không nên
columns_to_fill = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("Xử lý giá trị 0 bằng cách thay thế bằng trung bình:")
for col in columns_to_fill:
    mean_val = df[col][df[col] != 0].mean()
    df[col].replace(0, mean_val, inplace=True)
    print(f"  {col}: Thay 0 bằng {mean_val:.2f}")

print("\nSau khi xử lý:")
print((df == 0).sum())

In [ ]:
# Kiểm tra thống kê sau khi xử lý
print("Thống kê sau khi xử lý dữ liệu:")
print(df.describe())

In [ ]:
# Tách dữ liệu thành X (features) và y (target)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

print(f"Hình dạng X: {X.shape}")
print(f"Hình dạng y: {y.shape}")
print(f"\nCác đặc trưng:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i}. {col}")

In [ ]:
# Chia dữ liệu thành tập huấn luyện và tập kiểm tra (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Kích thước tập huấn luyện: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Kích thước tập kiểm tra: X_test {X_test.shape}, y_test {y_test.shape}")
print(f"\nTỷ lệ lớp trong tập huấn luyện:")
print(y_train.value_counts(normalize=True))
print(f"\nTỷ lệ lớp trong tập kiểm tra:")
print(y_test.value_counts(normalize=True))

In [ ]:
# Chuẩn hóa dữ liệu (Standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Trung bình của X_train_scaled (phải gần 0):")
print(np.round(X_train_scaled.mean(axis=0), 4))
print(f"\nĐộ lệch chuẩn của X_train_scaled (phải gần 1):")
print(np.round(X_train_scaled.std(axis=0), 4))

## 5. Xây dựng và Huấn luyện mô hình (Model Training)

### 5.1 Hướng dẫn (Guidance)
+ **Mô hình 1: Logistic Regression**
    - Mô hình tuyến tính đơn giản
    - Dễ diễn giải và nhanh
    - Phù hợp với bài toán phân loại nhị phân
+ **Mô hình 2: Random Forest** 
    - Mô hình ensemble (kết hợp nhiều cây)
    - Xử lý dữ liệu phi tuyến tốt
    - Có thể tính feature importance
    - Thường cho kết quả tốt nhất
+ **Mô hình 3: Support Vector Machine (SVM)**
    - Mô hình mạnh cho bài toán phân loại
    - Kernel RBF xử lý dữ liệu phi tuyến
    - Cần dữ liệu chuẩn hóa
+ **Nhận xét**:
    - So sánh Overfitting: RF train=99%, test=75% (overfitting) 
    - LR train=79%, test=71% (cân bằng hơn) 
    - Chọn mô hình dựa trên cân bằng giữa train/test accuracy

In [ ]:
# 5.1 Logistic Regression
print("1. LOGISTIC REGRESSION")

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

lr_train_pred = lr_model.predict(X_train_scaled)
lr_test_pred = lr_model.predict(X_test_scaled)

lr_train_acc = accuracy_score(y_train, lr_train_pred)
lr_test_acc = accuracy_score(y_test, lr_test_pred)

print(f"\nĐộ chính xác trên tập huấn luyện: {lr_train_acc:.4f}")
print(f"Độ chính xác trên tập kiểm tra: {lr_test_acc:.4f}")
print(f"\nBáo cáo phân loại:")
print(classification_report(y_test, lr_test_pred, target_names=['Không bệnh', 'Có bệnh']))

In [ ]:
# 5.2 Random Forest
print("2. RANDOM FOREST CLASSIFIER")

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10)
rf_model.fit(X_train_scaled, y_train)

rf_train_pred = rf_model.predict(X_train_scaled)
rf_test_pred = rf_model.predict(X_test_scaled)

rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

print(f"\nĐộ chính xác trên tập huấn luyện: {rf_train_acc:.4f}")
print(f"Độ chính xác trên tập kiểm tra: {rf_test_acc:.4f}")
print(f"\nBáo cáo phân loại:")
print(classification_report(y_test, rf_test_pred, target_names=['Không bệnh', 'Có bệnh']))

In [ ]:
# 5.3 Support Vector Machine (SVM)
print("3. SUPPORT VECTOR MACHINE (SVM)")

svm_model = SVC(kernel='rbf', probability=True, random_state=42, gamma='scale')
svm_model.fit(X_train_scaled, y_train)

svm_train_pred = svm_model.predict(X_train_scaled)
svm_test_pred = svm_model.predict(X_test_scaled)

svm_train_acc = accuracy_score(y_train, svm_train_pred)
svm_test_acc = accuracy_score(y_test, svm_test_pred)

print(f"\nĐộ chính xác trên tập huấn luyện: {svm_train_acc:.4f}")
print(f"Độ chính xác trên tập kiểm tra: {svm_test_acc:.4f}")
print(f"\nBáo cáo phân loại:")
print(classification_report(y_test, svm_test_pred, target_names=['Không bệnh', 'Có bệnh']))

## 6. Đánh giá mô hình (Model Evaluation)

### 6.1 Hướng dẫn (Guidance)
+ **Bước 1: So sánh Accuracy**
    - Accuracy = (TP + TN) / Total
    - Tính trên test set (dữ liệu chưa thấy)
    - Random Forest: 75.32% 
+ **Bước 2: Ma trận Nhầm Lẫn (Confusion Matrix)**
    ```
    ┌────────────────────────────────┐
    │      Dự đoán                   │
    │   0 (Không)  1 (Có)            │
    ├────────────────────────────────┤
    │  TN (đúng) │ FP (sai)│ Thực tế │
    │     83     │    17   │ 0(Không)│
    ├────────────────────────────────┤
    │  FN (sai)  │TP (đúng)│ Thực tế │
    │     21     │    33   │  1(Có)  │
    └────────────────────────────────┘
    ```
+ **Bước 3: AUC-ROC Score**
    - AUC = 0.5: tương đương random guess 
    - AUC = 0.7-0.8: mô hình tốt 
    - AUC = 0.8-0.9: mô hình rất tốt 
    - AUC = 1.0: mô hình hoàn hảo 
+ **Bước 4: Precision, Recall, F1-Score**
    - Precision: Trong những dự đoán "có bệnh", bao nhiêu % đúng
    - Recall: Trong những "có bệnh" thực tế, bao nhiêu % được phát hiện
    - F1-Score: Trung bình hài hòa của Precision và Recall
+ **Nhận xét**:
    - Random Forest AUC=0.819 (rất tốt) 
    - False Negative (FN) nguy hiểm hơn False Positive (FP) trong y tế 
    - Cần chọn threshold tùy theo mục tiêu bài toán

In [ ]:
# So sánh độ chính xác của các mô hình
models_comparison = {
    'Logistic Regression': {'Train': lr_train_acc, 'Test': lr_test_acc},
    'Random Forest': {'Train': rf_train_acc, 'Test': rf_test_acc},
    'SVM': {'Train': svm_train_acc, 'Test': svm_test_acc}
}

comparison_df = pd.DataFrame(models_comparison).T
print("\nSo sánh độ chính xác của các mô hình:")
print(comparison_df.to_string())

# Vẽ biểu đồ so sánh
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df['Train'], width, label='Tập Huấn Luyện', color='skyblue')
bars2 = ax.bar(x + width/2, comparison_df['Test'], width, label='Tập Kiểm Tra', color='salmon')

ax.set_xlabel('Mô hình', fontweight='bold')
ax.set_ylabel('Độ chính xác', fontweight='bold')
ax.set_title('So sánh độ chính xác giữa các mô hình', fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df.index)
ax.legend()
ax.set_ylim([0.7, 1.0])
ax.grid(alpha=0.3, axis='y')

# Thêm giá trị lên trên các cột
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Ma trận nhầm lẫn (Confusion Matrix)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models_list = [
    ('Logistic Regression', lr_test_pred),
    ('Random Forest', rf_test_pred),
    ('SVM', svm_test_pred)
]

for idx, (name, predictions) in enumerate(models_list):
    cm = confusion_matrix(y_test, predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Không bệnh', 'Có bệnh'],
                yticklabels=['Không bệnh', 'Có bệnh'],
                cbar_kws={'label': 'Số lượng'})
    axes[idx].set_title(f'Ma trận nhầm lẫn - {name}', fontweight='bold')
    axes[idx].set_xlabel('Dự đoán')
    axes[idx].set_ylabel('Thực tế')

plt.tight_layout()
plt.show()

In [ ]:
# Đường cong ROC (Receiver Operating Characteristic)
fig, ax = plt.subplots(figsize=(10, 7))

# Logistic Regression
lr_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_pred_proba)
lr_auc = roc_auc_score(y_test, lr_pred_proba)
ax.plot(lr_fpr, lr_tpr, label=f'Logistic Regression (AUC = {lr_auc:.4f})', linewidth=2)

# Random Forest
rf_pred_proba = rf_model.predict_proba(X_test_scaled)[:, 1]
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_pred_proba)
rf_auc = roc_auc_score(y_test, rf_pred_proba)
ax.plot(rf_fpr, rf_tpr, label=f'Random Forest (AUC = {rf_auc:.4f})', linewidth=2)

# SVM
svm_pred_proba = svm_model.predict_proba(X_test_scaled)[:, 1]
svm_fpr, svm_tpr, _ = roc_curve(y_test, svm_pred_proba)
svm_auc = roc_auc_score(y_test, svm_pred_proba)
ax.plot(svm_fpr, svm_tpr, label=f'SVM (AUC = {svm_auc:.4f})', linewidth=2)

# Đường tham chiếu
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)

ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('Đường cong ROC - So sánh các mô hình', fontweight='bold', fontsize=12)
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Tầm quan trọng của các đặc trưng (Feature Importance)
feature_importance = rf_model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("\nTầm quan trọng của các đặc trưng (Random Forest):")
print(importance_df.to_string())

# Vẽ biểu đồ
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
ax.set_xlabel('Mức độ quan trọng', fontweight='bold')
ax.set_title('Tầm quan trọng của các đặc trưng (Random Forest)', fontweight='bold', fontsize=12)
ax.invert_yaxis()
ax.grid(alpha=0.3, axis='x')

# Thêm giá trị lên thanh
for i, (importance) in enumerate(importance_df['Importance']):
    ax.text(importance + 0.005, i, f'{importance:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Dự đoán trên dữ liệu mới (Predictions on New Data)

In [ ]:
# Chọn mô hình tốt nhất (Random Forest có AUC cao nhất)
best_model = rf_model
best_scaler = scaler

print(f"Mô hình tốt nhất: Random Forest")
print(f"Độ chính xác trên tập kiểm tra: {rf_test_acc:.4f}")
print(f"AUC: {rf_auc:.4f}")

In [ ]:
# Ví dụ dự đoán trên bệnh nhân mới
# Các chỉ số: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age
new_patient_1 = pd.DataFrame({
    'Pregnancies': [2],
    'Glucose': [125],
    'BloodPressure': [78],
    'SkinThickness': [32],
    'Insulin': [88],
    'BMI': [27.5],
    'DiabetesPedigreeFunction': [0.35],
    'Age': [45]
})

print("BỆNH NHÂN 1 - Ví dụ dự đoán")
print(new_patient_1.to_string(index=False))

# Chuẩn hóa dữ liệu
new_patient_1_scaled = best_scaler.transform(new_patient_1)

# Dự đoán
prediction_1 = best_model.predict(new_patient_1_scaled)[0]
probability_1 = best_model.predict_proba(new_patient_1_scaled)[0]

result_text = 'Có bệnh tiểu đường  ' if prediction_1 == 1 else 'Không bệnh tiểu đường  '
print(f"\nKết quả dự đoán: {result_text}")
print(f"Xác suất Không bệnh: {probability_1[0]:.4f} ({probability_1[0]*100:.2f}%)")
print(f"Xác suất Có bệnh: {probability_1[1]:.4f} ({probability_1[1]*100:.2f}%)")

In [ ]:
# Dự đoán cho nhiều bệnh nhân
new_patients = pd.DataFrame({
    'Pregnancies': [2, 0, 5, 1, 3],
    'Glucose': [125, 90, 150, 110, 145],
    'BloodPressure': [78, 68, 95, 74, 88],
    'SkinThickness': [32, 20, 40, 25, 35],
    'Insulin': [88, 45, 120, 70, 100],
    'BMI': [27.5, 22.0, 32.5, 25.0, 30.0],
    'DiabetesPedigreeFunction': [0.35, 0.25, 0.5, 0.3, 0.45],
    'Age': [45, 28, 55, 35, 50]
})

print("DỰ ĐOÁN CHO NHIỀU BỆNH NHÂN")
print(new_patients.to_string(index=False))

# Chuẩn hóa và dự đoán
new_patients_scaled = best_scaler.transform(new_patients)
predictions = best_model.predict(new_patients_scaled)
probabilities = best_model.predict_proba(new_patients_scaled)

# Tạo bảng kết quả
results = pd.DataFrame({
    'Bệnh nhân': [f'BN {i+1}' for i in range(len(new_patients))],
    'Dự đoán': ['Có bệnh  ' if pred == 1 else 'Không bệnh  ' for pred in predictions],
    'P(Không bệnh)': [f"{prob[0]:.4f}" for prob in probabilities],
    'P(Có bệnh)': [f"{prob[1]:.4f}" for prob in probabilities]
})

print("\nKết quả dự đoán:")
print(results.to_string(index=False))

## 8. Lưu mô hình (Save Models)

In [ ]:
# Tạo thư mục để lưu mô hình
save_dir = 'diabetes_models'
os.makedirs(save_dir, exist_ok=True)

# Lưu các mô hình
joblib.dump(lr_model, f'{save_dir}/logistic_regression_model.joblib')
joblib.dump(rf_model, f'{save_dir}/random_forest_model.joblib')
joblib.dump(svm_model, f'{save_dir}/svm_model.joblib')

# Lưu scaler
joblib.dump(scaler, f'{save_dir}/scaler.joblib')

print(f"✓ Các mô hình đã được lưu vào thư mục '{save_dir}'")
print(f"\nCác file được lưu:")
for i, file in enumerate(os.listdir(save_dir), 1):
    print(f"  {i}. {file}")

In [ ]:
# Lưu các tính năng và thông tin mô hình
model_info = {
    'features': list(X.columns),
    'feature_importance': dict(zip(X.columns, rf_model.feature_importances_.tolist())),
    'model_performance': {
        'Logistic Regression': {
            'train_accuracy': float(lr_train_acc),
            'test_accuracy': float(lr_test_acc),
            'auc': float(lr_auc)
        },
        'Random Forest': {
            'train_accuracy': float(rf_train_acc),
            'test_accuracy': float(rf_test_acc),
            'auc': float(rf_auc)
        },
        'SVM': {
            'train_accuracy': float(svm_train_acc),
            'test_accuracy': float(svm_test_acc),
            'auc': float(svm_auc)
        }
    },
    'best_model': 'Random Forest',
    'total_samples': len(df),
    'training_samples': len(X_train),
    'test_samples': len(X_test)
}

import json
with open(f'{save_dir}/model_info.json', 'w', encoding='utf-8') as f:
    json.dump(model_info, f, ensure_ascii=False, indent=4)

print("Thông tin mô hình đã được lưu vào 'model_info.json'")

## 9. Tải và sử dụng mô hình đã lưu (Load and Use Saved Models)

In [ ]:
# Tải mô hình Random Forest (mô hình tốt nhất)
loaded_model = joblib.load(f'{save_dir}/random_forest_model.joblib')
loaded_scaler = joblib.load(f'{save_dir}/scaler.joblib')

print("Mô hình đã được tải thành công!")

# Kiểm tra bằng cách dự đoán trên tập kiểm tra
test_pred = loaded_model.predict(loaded_scaler.transform(X_test))
test_acc = accuracy_score(y_test, test_pred)
print(f"Độ chính xác trên tập kiểm tra: {test_acc:.4f}")

In [ ]:
# Tải và hiển thị thông tin mô hình
with open(f'{save_dir}/model_info.json', 'r', encoding='utf-8') as f:
    loaded_info = json.load(f)

print("THÔNG TIN MÔ HÌNH ĐƯỢC LƯU")

print(f"\nCác đặc trưng ({len(loaded_info['features'])}) :")
for i, feature in enumerate(loaded_info['features'], 1):
    print(f"  {i}. {feature}")

print(f"\nTầm quan trọng đặc trưng (Random Forest):")
sorted_importance = sorted(loaded_info['feature_importance'].items(), key=lambda x: x[1], reverse=True)
for feature, importance in sorted_importance:
    print(f"  {feature}: {importance:.4f}")

print(f"\nHiệu suất mô hình:")
for model_name, metrics in loaded_info['model_performance'].items():
    print(f"\n  {model_name}:")
    print(f"    - Train Accuracy: {metrics['train_accuracy']:.4f}")
    print(f"    - Test Accuracy: {metrics['test_accuracy']:.4f}")
    print(f"    - AUC: {metrics['auc']:.4f}")

print(f"\nThông tin dữ liệu:")
print(f"  - Tổng mẫu: {loaded_info['total_samples']}")
print(f"  - Mẫu huấn luyện: {loaded_info['training_samples']}")
print(f"  - Mẫu kiểm tra: {loaded_info['test_samples']}")
print(f"\nMô hình tốt nhất: {loaded_info['best_model']}")

## 10. Tóm tắt và Kết luận (Summary & Conclusion)

In [ ]:
print("TÓM TẮT DỰ ÁN - PHÂN LOẠI BỆNH TIỂU ĐƯỜNG")

print("\n1. DỮ LIỆU:")
print(f"   - Bộ dữ liệu: Pima Indians Diabetes Dataset")
print(f"   - Số mẫu: {len(df)}")
print(f"   - Số đặc trưng: {len(X.columns)}")
print(f"   - Số lớp: 2 (Không bệnh/Có bệnh)")
print(f"   - Tỷ lệ lớp:")
for outcome, count in y.value_counts().sort_index().items():
    print(f"     * {'Không bệnh' if outcome == 0 else 'Có bệnh'}: {count} ({count/len(y)*100:.1f}%)")

print(f"\n2. TIỀN XỬ LÝ:")
print(f"   - Xử lý giá trị 0 không hợp lý bằng giá trị trung bình")
print(f"   - Chuẩn hóa dữ liệu (StandardScaler)")
print(f"   - Chia dữ liệu: 80% huấn luyện, 20% kiểm tra")

print(f"\n3. MÔ HÌNH:")
print(f"   ✓ Logistic Regression (Hồi quy logistic)")
print(f"     - Đơn giản, dễ diễn giải")
print(f"     - Độ chính xác: {lr_test_acc:.4f}")
print(f"   ✓ Random Forest (Rừng ngẫu nhiên) ")
print(f"     - Mô hình ensemble, hiệu suất cao")
print(f"     - Độ chính xác: {rf_test_acc:.4f}")
print(f"   ✓ Support Vector Machine (SVM)")
print(f"     - Mô hình mạnh mẽ với kernel RBF")
print(f"     - Độ chính xác: {svm_test_acc:.4f}")

print(f"\n4. KẾT QUẢ:")
best_acc = max(lr_test_acc, rf_test_acc, svm_test_acc)
best_model_name = 'Logistic Regression' if best_acc == lr_test_acc else ('Random Forest' if best_acc == rf_test_acc else 'SVM')
print(f"   - Mô hình tốt nhất: {best_model_name}")
print(f"   - Độ chính xác cao nhất: {best_acc:.4f}")
print(f"   - AUC cao nhất: {rf_auc:.4f} (Random Forest)")

print(f"\n5. ĐẶC TRƯNG QUAN TRỌNG NHẤT:")
top_features = importance_df.head(3)
for idx, (_, row) in enumerate(top_features.iterrows(), 1):
    print(f"   {idx}. {row['Feature']}: {row['Importance']:.4f}")

print(f"\n6. ỨNG DỤNG:")
print(f"   - Dự đoán nguy cơ bệnh tiểu đường cho bệnh nhân")
print(f"   - Hỗ trợ chẩn đoán sớm")
print(f"   - Xác định nhóm bệnh nhân cần theo dõi")

print(f"\n7. FILE ĐƯỢC LƯU:")
print(f"   - logistic_regression_model.joblib")
print(f"   - random_forest_model.joblib")
print(f"   - svm_model.joblib")
print(f"   - scaler.joblib")
print(f"   - model_info.json")
